In [ ]:
"""Baselines scored on the SAME outer folds as the MoE, with paired AUC differences (95% CI, no p-values)."""
import os
import numpy as np
import pandas as pd
from joblib import Parallel, delayed
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression, LogisticRegressionCV
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from help.stats_utils import SEED, N_REPEATS, outer_splits, bootstrap, youden_threshold

In [ ]:
OUT_DIR, POP_PATH = "Results_MoE", "analytic_population.pkl"
C_GRID = [1e-4, 1e-3, 1e-2, 1e-1, 1.0, 10.0]          # same grid as 02_train
RUN_XGB = True
USE_CACHE = False                                       

inner_cv = lambda seed=SEED: StratifiedKFold(5, shuffle=True, random_state=seed) 

oof = pd.read_csv(f"{OUT_DIR}/oof_predictions.csv")
pop = pd.read_pickle(POP_PATH)
feat_cols = pop.attrs["feat_cols"]

# Extra non-vocal columns needed for the demographic+lifestyle+psychological baseline.
DEMO_LIFESTYLE_PSYCH_COLS = [
    "age",
    "is_smoker",
    "has_depression",
    "gad_score",
    "pss_score"]

In [ ]:
COUNTRY_COL = "country_group"      
extra_cols = [c for c in DEMO_LIFESTYLE_PSYCH_COLS if c != "age"] + [COUNTRY_COL]

d = oof[["participant_id", "gender", "y_any", "predicted_composite_score", "flag_rate_composite"]].merge(
    pop[["participant_id", "age", *feat_cols, *extra_cols]], on="participant_id", how="left")  # keeps 02_train's row order
assert len(d) == len(oof) and d["age"].notna().all()

# Sanity check on the score
cs = d["predicted_composite_score"]
assert cs.between(0, 100).all(), "predicted_composite_score is not on a 0-100 scale as assumed"

missing_n = d[DEMO_LIFESTYLE_PSYCH_COLS + [COUNTRY_COL]].isna().any(axis=1).sum()
if missing_n:
    print(f"WARNING: {missing_n} participants have missing values in the non-vocal baseline columns")

In [ ]:
MODELS = {
    "Age only": (["age"], lambda y_tr=None, rep=0: make_pipeline(StandardScaler(), LogisticRegression(class_weight="balanced", max_iter=2000))),
    "Voice PCA + LR": (feat_cols, lambda y_tr=None, rep=0: Pipeline([("scale", StandardScaler()), ("pca", PCA(0.95, random_state=SEED)),
        ("lr", LogisticRegressionCV(Cs=C_GRID, cv=inner_cv(SEED + rep), scoring="roc_auc", class_weight="balanced", max_iter=2000))])),
}
if RUN_XGB:
    from xgboost import XGBClassifier

    def make_xgb(y_tr=None, rep=0):
        # scale_pos_weight = n_negative / n_positive on THIS outer-training fold, mirroring
        # class_weight="balanced" used by every LR baseline. Without this, XGBoost is scored
        # on an uneven playing field relative to the other models whenever y_any is imbalanced.
        if y_tr is None:
            spw = 1.0
        else:
            y_tr = np.asarray(y_tr)
            n_pos = y_tr.sum()
            n_neg = len(y_tr) - n_pos
            spw = n_neg / max(n_pos, 1)
        return GridSearchCV(
            XGBClassifier(subsample=0.8, colsample_bytree=0.3, learning_rate=0.05, eval_metric="logloss",
                          random_state=SEED + rep, n_jobs=1, scale_pos_weight=spw),
            {"max_depth": [2, 4], "n_estimators": [200, 400]}, cv=3, scoring="roc_auc")

    MODELS["Voice XGBoost"] = (feat_cols, make_xgb)

def make_demo_lifestyle_psych_pipeline(y_tr=None, rep=0):
    pre = ColumnTransformer([
        ("num", StandardScaler(), DEMO_LIFESTYLE_PSYCH_COLS),
        ("cat", OneHotEncoder(handle_unknown="ignore"), [COUNTRY_COL]),
    ])
    return Pipeline([
        ("pre", pre),
        ("lr", LogisticRegressionCV(Cs=C_GRID, cv=inner_cv(SEED + rep), scoring="roc_auc",
                                     class_weight="balanced", max_iter=2000))])

MODELS["Demographic + lifestyle + psychological"] = (
    [*DEMO_LIFESTYLE_PSYCH_COLS, COUNTRY_COL],
    make_demo_lifestyle_psych_pipeline)

In [ ]:
def fit_fold(make_model, X, y, tr, te, rep):                              
    Xtr = X.iloc[tr] if hasattr(X, "iloc") else X[tr]
    Xte = X.iloc[te] if hasattr(X, "iloc") else X[te]
    model = make_model(y_tr=y[tr], rep=rep).fit(Xtr, y[tr])               
    return model.predict_proba(Xte)[:, 1], model.predict_proba(Xtr)[:, 1]

def oof_scores_and_flags(make_model, X, y):
    splits = list(outer_splits(y))
    out = Parallel(n_jobs=-1)(delayed(fit_fold)(make_model, X, y, tr, te, rep) for rep, tr, te in splits)  

    S = np.zeros((N_REPEATS, len(y)))
    F = np.zeros((N_REPEATS, len(y)))
    for (rep, tr, te), (p_te, p_tr) in zip(splits, out):
        S[rep, te] = p_te
        thr = youden_threshold(y[tr], p_tr)
        F[rep, te] = (p_te >= thr).astype(float)

    return S.mean(0), F.mean(0)

def perf(y_, s_):
    """AUC + sens/spec at a Youden threshold re-estimated on the same scores"""
    pred = s_ >= youden_threshold(y_, s_)
    return {"AUC": roc_auc_score(y_, s_), "Sensitivity": pred[y_ == 1].mean(), "Specificity": (~pred[y_ == 0]).mean()}

def d_auc(y_, a, b):
    return {"AUC difference (MoE - model)": roc_auc_score(y_, a) - roc_auc_score(y_, b)}


metrics, diffs = [], []
for sex, g in d.groupby("gender"):
    y = g["y_any"].to_numpy().astype(int)
    scores = {"MoE": g["predicted_composite_score"].to_numpy() / 100}
    flags  = {"MoE": g["flag_rate_composite"].to_numpy()}   
    for name, (cols, make) in MODELS.items():
        cache = f"{OUT_DIR}/cache_{sex}_{name.replace(' ', '_').replace('+', '')}.npy"
        cache_f = f"{OUT_DIR}/cacheflag_{sex}_{name.replace(' ', '_').replace('+', '')}.npy"
        if USE_CACHE and os.path.exists(cache) and os.path.exists(cache_f):
            scores[name] = np.load(cache)
            flags[name] = np.load(cache_f)
        else:
            X_input = g[cols] if COUNTRY_COL in cols else g[cols].to_numpy(float)
            scores[name], flags[name] = oof_scores_and_flags(make, X_input, y)
            np.save(cache, scores[name])
            np.save(cache_f, flags[name])
    for name, s in scores.items():
        def perf_byfold(y_, s_, f_):
            return {"AUC": roc_auc_score(y_, s_),
                    "Sensitivity": f_[y_ == 1].mean(), "Specificity": (1 - f_[y_ == 0]).mean()}
        metrics.append(bootstrap(perf_byfold, y, s, flags[name]).assign(gender=sex, model=name))
        if name != "MoE":
            tab = bootstrap(d_auc, y, scores["MoE"], s)
            ci_lo, ci_hi = tab["ci_lo"].iloc[0], tab["ci_hi"].iloc[0]
            verdict = "MoE better" if ci_lo > 0 else "model better" if ci_hi < 0 else "inconclusive"
            diffs.append(tab.assign(gender=sex, model=name, verdict=verdict))

metrics, diffs = pd.concat(metrics, ignore_index=True), pd.concat(diffs, ignore_index=True)
metrics.to_csv(f"{OUT_DIR}/baseline_metrics.csv", index=False)
diffs.to_csv(f"{OUT_DIR}/baseline_auc_difference.csv", index=False)

In [ ]:
fmt = lambda r: f"{r.estimate:.3f} ({r.ci_lo:.3f}-{r.ci_hi:.3f})"
metrics["cell"] = metrics.apply(fmt, axis=1)
order = ["MoE", *MODELS]
tbl = (metrics.pivot_table(index=["gender", "model"], columns="metric", values="cell", aggfunc="first")
       [["AUC", "Sensitivity", "Specificity"]].reindex(order, level="model"))
print("\nTable A. Out-of-fold performance (sens/spec: Youden threshold on the same OOF scores for every model, apparent)")
print(tbl.to_string())
tbl.to_csv(f"{OUT_DIR}/baseline_table.csv")

diffs["AUC diff (95% CI)"] = diffs.apply(fmt, axis=1)
print("\nTable B. Paired AUC difference, MoE - baseline (same participants, same folds)")
print(diffs[["gender", "model", "AUC diff (95% CI)", "verdict"]].to_string(index=False))